# NB-04 — EDA Estratificación e Indicadores de Hábitat
## DataJam Bogotá 2026

**Objetivo:** Explorar `Esoc.csv` (84 MB), `manzanaestratificacion.json`, `predioruralestrato.json` e `indicadores-urbanos-habitat-en-cifras-en-las-localidades.xlsx`.

**Hipótesis central:** El estrato socioeconómico es el predictor clave de la vulnerabilidad en manejo de residuos y emergencias.

**Secciones:**
1. Exploración de `Esoc.csv` — distribución de estratos por lote
2. Exploración de `manzanaestratificacion.json`
3. Exploración de `predioruralestrato.json`
4. Exploración de indicadores de hábitat (XLSX)
5. Agregación exploratoria de estrato por UPZ
6. Tabla de decisiones

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
BRONZE = os.path.join(BASE, 'Bronze')

sns.set_theme(style='whitegrid', palette='muted')
print(f'Base: {BASE}')

## 1. Exploración de Esoc.csv — Estrato por Lote

In [ ]:
ESOC_PATH = os.path.join(BRONZE, 'Esoc.csv')
size_mb = os.path.getsize(ESOC_PATH) / 1024**2
print(f'Tamaño: {size_mb:.1f} MB')

# Leer en chunks para eficiencia con archivo de 84 MB
print('Leyendo Esoc.csv en chunks...')
chunks = []
for chunk in pd.read_csv(ESOC_PATH, chunksize=200000, low_memory=False):
    chunks.append(chunk)
df_esoc = pd.concat(chunks, ignore_index=True)
print(f'Filas: {len(df_esoc):,} | Columnas: {df_esoc.shape[1]}')
print(f'\nColumnas: {list(df_esoc.columns)}')
print(f'\nTipos de datos:')
print(df_esoc.dtypes)
print(f'\nPrimeras 3 filas:')
df_esoc.head(3)


In [ ]:
# Identificar columna de estrato
col_estrato = [c for c in df_esoc.columns if 'strato' in c.lower() or 'ESTRATO' in c.upper()]
col_lote = [c for c in df_esoc.columns if 'lote' in c.lower() or 'CLote' in c]
col_chip = [c for c in df_esoc.columns if 'chip' in c.lower() or 'Chip' in c]

print(f'Columna de estrato: {col_estrato}')
print(f'Columna de lote: {col_lote}')
print(f'Columna de chip: {col_chip}')

if col_estrato:
    col_e = col_estrato[0]
    estrato_dist = df_esoc[col_e].value_counts().sort_index()
    print(f'\nDistribución de estratos:')
    print(estrato_dist)
    print(f'\nPorcentaje por estrato:')
    print((estrato_dist / len(df_esoc) * 100).round(2))
    print(f'\nNulos en estrato: {df_esoc[col_e].isna().sum():,} ({df_esoc[col_e].isna().mean()*100:.1f}%)')

In [ ]:
if col_estrato:
    col_e = col_estrato[0]
    # Convertir a numérico
    df_esoc[col_e] = pd.to_numeric(df_esoc[col_e], errors='coerce')
    
    # Calcular distribución directamente (sin filtrar el df entero)
    dist = df_esoc[col_e].value_counts().sort_index()
    dist_valida = dist[dist.index.isin([1,2,3,4,5,6])]
    print('Distribución de estratos (conteos):')
    print(dist_valida)
    print(f'\nEstratos fuera de rango [1-6]:')
    print(dist[~dist.index.isin([1,2,3,4,5,6])])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4','#313695']
    bars = axes[0].bar(dist_valida.index.astype(str), dist_valida.values,
                       color=colors[:len(dist_valida)], edgecolor='white')
    axes[0].set_title('Distribución de Estratos en Esoc.csv\n(Por número de lotes)',
                      fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Estrato')
    axes[0].set_ylabel('Número de lotes')
    for bar in bars:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                    f'{int(bar.get_height()):,}', ha='center', fontsize=8)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    
    pct = (dist_valida / len(df_esoc) * 100).round(1)
    axes[1].pie(pct.values,
               labels=[f'Estrato {int(e)}\n({p}%)' for e, p in zip(pct.index, pct.values)],
               colors=colors[:len(pct)], startangle=90, pctdistance=0.85)
    axes[1].set_title('Proporción de Lotes por Estrato\n(Bogotá D.C.)',
                      fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../outputs/04_distribucion_estratos_esoc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gráfica guardada.')
else:
    print('Columna de estrato no identificada')


## 2. Exploración de manzanaestratificacion.json

In [ ]:
MANZANA_PATH = os.path.join(BRONZE, 'manzanaestratificacion.json')
size_mb = os.path.getsize(MANZANA_PATH) / 1024**2
print(f'Tamaño: {size_mb:.1f} MB')

# Leer primeros bytes para identificar estructura
print('\nInspeccionando estructura del archivo...')
with open(MANZANA_PATH, 'r', encoding='utf-8', errors='replace') as f:
    inicio = f.read(3000)
print(f'Inicio: {inicio[:800]}')

In [ ]:
# Intentar leer como GeoJSON con geopandas (muestra)
try:
    print('Intentando leer como GeoJSON (muestra 200 features)...')
    gdf_manzana = gpd.read_file(MANZANA_PATH, rows=200, engine='pyogrio')
    print(f'✅ Es GeoJSON | Tipo geom: {gdf_manzana.geom_type.value_counts().idxmax()}')
    print(f'Columnas: {list(gdf_manzana.columns)}')
    print(f'CRS: {gdf_manzana.crs}')
    print(f'\nPrimeras 3 filas (sin geometría):')
    print(gdf_manzana.drop(columns='geometry').head(3))
    
    # Buscar columna de estrato
    col_est = [c for c in gdf_manzana.columns if 'strato' in c.lower() or 'ESTRATO' in c.upper()]
    print(f'\nColumnas de estrato encontradas: {col_est}')
except Exception as e:
    print(f'❌ No es GeoJSON estándar: {e}')
    print('Intentando leer como JSON plano...')
    try:
        # Leer parcialmente para no sobrecargar memoria
        import json
        with open(MANZANA_PATH, 'r', encoding='utf-8', errors='replace') as f:
            data = json.load(f)
        print(f'Tipo de datos: {type(data)}')
        if isinstance(data, dict):
            print(f'Keys: {list(data.keys())}')
        elif isinstance(data, list):
            print(f'Lista de {len(data)} elementos')
            print(f'Primer elemento: {data[0] if data else "vacío"}')
    except Exception as e2:
        print(f'Error leyendo como JSON: {e2}')

## 3. Exploración de predioruralestrato.json

In [ ]:
RURAL_PATH = os.path.join(BRONZE, 'predioruralestrato.json')
size_mb = os.path.getsize(RURAL_PATH) / 1024**2
print(f'predioruralestrato.json — Tamaño: {size_mb:.1f} MB')

try:
    gdf_rural = gpd.read_file(RURAL_PATH, rows=50, engine='pyogrio')
    print(f'Tipo geom: {gdf_rural.geom_type.value_counts().idxmax()}')
    print(f'Columnas: {list(gdf_rural.columns)}')
    print(f'CRS: {gdf_rural.crs}')
    
    # Verificar si está dentro de Bogotá D.C.
    bbox = gdf_rural.total_bounds
    print(f'Bounding box (lon/lat): {bbox}')
    # Bogotá D.C. aprox: -74.3 a -73.9 lon, 4.4 a 4.9 lat
    es_bogota = (-74.5 < bbox[0] < -73.5) and (4.0 < bbox[1] < 5.0)
    print(f'¿Está en coordenadas de Bogotá?: {es_bogota}')
except Exception as e:
    print(f'Error: {e}')

## 4. Exploración de Indicadores de Hábitat (XLSX)

In [ ]:
HABITAT_PATH = os.path.join(BRONZE, 'indicadores-urbanos-habitat-en-cifras-en-las-localidades.xlsx')

print('=== INDICADORES DE HÁBITAT ===')
xf = pd.ExcelFile(HABITAT_PATH)
print(f'Hojas disponibles: {xf.sheet_names}')

for sheet in xf.sheet_names:
    df_hab = pd.read_excel(HABITAT_PATH, sheet_name=sheet)
    print(f'\n--- Hoja: "{sheet}" ---')
    print(f'  Shape: {df_hab.shape}')
    print(f'  Columnas: {list(df_hab.columns)}')
    print(f'  Primeras 3 filas:')
    print(df_hab.head(3).to_string())

In [ ]:
# Leer la hoja principal y buscar campos de población y localidad
df_hab = pd.read_excel(HABITAT_PATH, sheet_name=0)
df_hab.columns = [str(c).strip() for c in df_hab.columns]

# Identificar columna de localidad
col_loc = [c for c in df_hab.columns if 'localidad' in c.lower() or 'local' in c.lower()]
col_pob = [c for c in df_hab.columns if 'poblac' in c.lower() or 'habitant' in c.lower()]
col_area = [c for c in df_hab.columns if 'area' in c.lower() or 'área' in c.lower() or 'km' in c.lower()]

print(f'Columnas de localidad: {col_loc}')
print(f'Columnas de población: {col_pob}')
print(f'Columnas de área: {col_area}')
print(f'\nTodas las columnas:')
for c in df_hab.columns:
    print(f'  - {c}: {df_hab[c].dtype} | nulos: {df_hab[c].isna().sum()}')

In [ ]:
# Visualizar distribución de indicadores por localidad (si hay datos de población)
if col_pob:
    col_p = col_pob[0]
    col_l = col_loc[0] if col_loc else df_hab.columns[0]
    
    df_plot = df_hab[[col_l, col_p]].dropna().copy()
    df_plot[col_p] = pd.to_numeric(df_plot[col_p], errors='coerce')
    df_plot = df_plot.dropna().sort_values(col_p, ascending=False).head(20)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(df_plot[col_l].astype(str).tolist()[::-1],
            df_plot[col_p].tolist()[::-1], color='steelblue', edgecolor='white')
    ax.set_title(f'{col_p} por Localidad', fontsize=12, fontweight='bold')
    ax.set_xlabel(col_p)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.savefig('../outputs/04_indicadores_habitat_localidad.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No se encontró columna de población — revisar XLSX manualmente')
    print(df_hab.head(10).to_string())

## 5. Agregación Exploratoria — Estrato Modal por UPZ

In [ ]:
# Exploramos si es posible agregar Esoc.csv a nivel UPZ
# Para eso necesitamos un campo de join entre lotes y UPZ

print('=== ANÁLISIS DE CAMPOS JOIN: Esoc.csv → UPZ ===')
print()
print('Columnas en Esoc.csv:')
print(df_esoc.columns.tolist())
print()
print('Muestra de valores ESoCLote (código de lote):')
if 'ESoCLote' in df_esoc.columns:
    muestra_lote = df_esoc['ESoCLote'].dropna().head(10).tolist()
    print(muestra_lote)
    print()
    # El código de lote IDECA tiene estructura:
    # LLLLLL0CC0PP donde L=localidad(6), C=manzana(2), P=predio(2)
    # Los primeros 2 dígitos generalmente = código de localidad
    if len(str(muestra_lote[0])) >= 2:
        cod_localidades = df_esoc['ESoCLote'].astype(str).str[:2].value_counts().head(25)
        print('Distribución de primeros 2 dígitos (posible código localidad):')
        print(cod_localidades)
else:
    print('Columna ESoCLote no encontrada — revisar nombre exacto de columnas')
    print(df_esoc.iloc[0])

In [ ]:
if 'ESoCLote' in df_esoc.columns and 'ESoEstrato' in df_esoc.columns:
    df_esoc['cod_localidad_aprox'] = df_esoc['ESoCLote'].astype(str).str[:2]
    df_esoc['ESoEstrato_num'] = pd.to_numeric(df_esoc['ESoEstrato'], errors='coerce')
    
    # Agrupar directamente sin crear df_valido en memoria
    estrato_por_localidad = (
        df_esoc[df_esoc['ESoEstrato_num'].between(1, 6)]
        .groupby('cod_localidad_aprox')['ESoEstrato_num']
        .agg(lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan)
        .reset_index()
        .rename(columns={'ESoEstrato_num': 'estrato_modal'})
    )
    
    print('=== ESTRATO MODAL POR LOCALIDAD ===')
    print(estrato_por_localidad.sort_values('cod_localidad_aprox').to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4','#313695']
    est_plot = estrato_por_localidad.dropna().sort_values('cod_localidad_aprox')
    bar_colors = [colors[int(e)-1] if 1 <= int(e) <= 6 else 'gray'
                  for e in est_plot['estrato_modal']]
    ax.bar(est_plot['cod_localidad_aprox'], est_plot['estrato_modal'],
           color=bar_colors, edgecolor='white')
    ax.set_title('Estrato Modal por Código de Localidad (Esoc.csv)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Código de Localidad')
    ax.set_ylabel('Estrato Modal')
    ax.set_yticks([1, 2, 3, 4, 5, 6])
    plt.tight_layout()
    plt.savefig('../outputs/04_estrato_modal_localidad.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Guardar para Silver
    estrato_por_localidad.to_csv('../Silver/estrato_modal_por_localidad.csv', index=False)
    print('Guardado en Silver/estrato_modal_por_localidad.csv')
else:
    print('Columnas ESoCLote o ESoEstrato no encontradas')


## 6. Tabla de Decisiones — Fuentes de Estratificación

In [ ]:
decisiones_estrato = [
    {
        'fuente': 'Esoc.csv',
        'descripcion': 'Estrato por lote (ESoCLote, ESoChip, ESoEstrato)',
        'cobertura': 'Todos los predios urbanos de Bogotá',
        'nivel_agregacion': 'Lote → agregar a UPZ/Localidad',
        'join_key': 'ESoCLote (código de lote IDECA)',
        'decision': '✅ USAR — Fuente oficial IDECA. Agregar estrato modal a nivel localidad/UPZ.',
        'ventajas': 'Mayor detalle, fuente oficial'
    },
    {
        'fuente': 'manzanaestratificacion.json',
        'descripcion': 'Estratificación por manzana (GeoJSON)',
        'cobertura': 'Manzanas urbanas Bogotá',
        'nivel_agregacion': 'Manzana → agregar a UPZ',
        'join_key': 'Código de manzana o spatial join',
        'decision': '⚠️ EVALUAR — Muy pesado (83 MB). Usar solo si Esoc.csv no tiene join directo con UPZ.',
        'ventajas': 'Ya georreferenciado, permite spatial join'
    },
    {
        'fuente': 'predioruralestrato.json',
        'descripcion': 'Predios rurales con estrato',
        'cobertura': 'Zona rural de Bogotá',
        'nivel_agregacion': 'Predio rural',
        'join_key': 'N/A',
        'decision': '❌ DESCARTAR — El análisis es urbano. Las zonas rurales están fuera del foco.',
        'ventajas': 'N/A'
    },
    {
        'fuente': 'indicadores_habitat.xlsx',
        'descripcion': 'Indicadores urbanos por localidad',
        'cobertura': '20 localidades de Bogotá',
        'nivel_agregacion': 'Localidad',
        'join_key': 'Nombre o código de localidad',
        'decision': '✅ USAR — Incluye indicadores de contexto: población, área, etc.',
        'ventajas': 'Agregado listo para usar a nivel localidad'
    }
]

dec_est_df = pd.DataFrame(decisiones_estrato)
print('=== TABLA DE DECISIONES — FUENTES DE ESTRATIFICACIÓN ===')
print(dec_est_df[['fuente','decision']].to_string(index=False))

dec_est_df.to_csv('../Silver/decisiones_estratificacion.csv', index=False)
print('\n✅ NB-04 completado. Guardado en Silver/decisiones_estratificacion.csv')
print('\n=== RESUMEN FINAL DE EXPLORACIÓN BRONZE ===')
print('Siguiente paso: ejecutar scripts Silver para limpiar y unificar los datos seleccionados.')